In [1]:
# --- 0. Environment Setup and Library Installations (Run this cell FIRST) ---
# This cell handles all library installations and initial setup.
# It's crucial to run this, then RESTART RUNTIME, and then run all cells.

print("Starting environment setup...")

# Ensure pip is up-to-date
!pip install --upgrade pip

# Aggressive uninstall to remove any lingering conflicting versions
print("\nUninstalling existing transformers, sentence-transformers, huggingface_hub, accelerate...")
!pip uninstall -y transformers sentence-transformers huggingface_hub accelerate

# Clean pip cache to ensure fresh downloads
print("\nClearing pip cache...")
!pip cache purge

# Install known compatible and stable versions to avoid recent breaking changes
# These versions are typically very stable and widely used.
print("\nInstalling specific stable versions: transformers==4.30.0, sentence-transformers==2.2.2, huggingface_hub==0.16.4, accelerate==0.21.0...")
!pip install transformers==4.30.0 # Stable version
!pip install sentence-transformers==2.2.2 # Compatible with transformers 4.30.0
!pip install huggingface_hub==0.16.4 # Known stable version that has 'cached_download'
!pip install accelerate==0.21.0 # Compatible with transformers 4.30.0

# Verify installed versions
print("\n--- Verifying installed versions ---")
!pip show transformers
!pip show sentence-transformers
!pip show huggingface_hub
!pip show accelerate

print("\n--- Installation commands executed ---")
print("!!! IMPORTANT: You MUST RESTART YOUR COLAB RUNTIME now !!!")
print("Go to 'Runtime' -> 'Restart runtime' in the top menu.")
print("After the runtime restarts, run ALL cells from the very beginning.")

# --- END OF ENVIRONMENT SETUP ---

# Once the runtime is restarted, the rest of your code can be pasted below.
# All necessary imports should now work.

import pandas as pd
import os
import openai
from google.colab import userdata # Correct import for Colab secrets
import json
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import warnings
import numpy as np
import time # Import time for delays
# You'll need to mount your Google Drive if your data is there
from google.colab import drive
drive.mount('/content/drive')


# --- 1. Configuration and Setup ---
project_path = '/content/drive/MyDrive/MoE'
excel_data_path = os.path.join(project_path, 'My Data.xlsx') # Path to your data file
ablation_output_excel_path = os.path.join(project_path, "Ablation_Study_Results.xlsx")

if not os.path.exists(project_path):
    print(f"Error: Project path '{project_path}' does not exist. Please create it or adjust the path.")
    try:
        os.makedirs(project_path)
        print(f"Created project directory: {project_path}")
    except Exception as e:
        print(f"Failed to create project directory: {e}. Please create it manually.")
        exit()

# Initialize SentenceTransformer model for embeddings
try:
    embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
    print("SentenceTransformer model loaded successfully.")
except Exception as e:
    print(f"Error loading SentenceTransformer model: {e}. "
          "Automated similarity scores may not work correctly.")
    embedding_model = None

# --- 2. Load Data and Build Vector Store ---
def load_data_and_build_vector_store(excel_path, embedding_model):
    print(f"Attempting to load data from: {excel_path}")
    if not os.path.exists(excel_path):
        print(f"Error: Data file '{excel_path}' not found. Please ensure it's in the correct path.")
        return [], None

    try:
        df = pd.read_excel(excel_path)
        print(f"Loaded {len(df)} rows from '{excel_path}'.")

        if 'non-inclusive' not in df.columns or 'inclusive' not in df.columns:
            print("Error: 'non-inclusive' or 'inclusive' columns not found in the Excel file.")
            return [], None

        contexts = []
        embeddings = []
        for index, row in df.iterrows():
            original_sentence = str(row['non-inclusive']).strip()
            revised_sentence = str(row['inclusive']).strip()
            # Ensure category is handled correctly, defaulting if not present
            category = str(row.get('category', 'N/A')).strip()

            if original_sentence and revised_sentence and embedding_model:
                try:
                    embedding = embedding_model.encode(original_sentence, convert_to_tensor=False)
                    embeddings.append(embedding)
                    contexts.append({
                        "original": original_sentence,
                        "revised": revised_sentence,
                        "category": category
                    })
                except Exception as e:
                    print(f"Warning: Could not create embedding for row {index}: {e}")
            elif not original_sentence or not revised_sentence:
                print(f"Warning: Skipping row {index} due to empty 'non-inclusive' or 'inclusive' content.")

        embeddings_array = np.array(embeddings) if embeddings else None
        print(f"Generated embeddings for {len(embeddings)} sentences.")
        return contexts, embeddings_array

    except Exception as e:
        print(f"Error loading or processing Excel data: {e}")
        return [], None

# Load your data and build the vector store when the script starts
all_contexts_data, all_embeddings_array = load_data_and_build_vector_store(excel_data_path, embedding_model)
if not all_contexts_data or all_embeddings_array is None:
    print("WARNING: Vector store could not be initialized from My Data.xlsx. Experts will run without dynamic few-shot examples.")
    # Fallback to dummy contexts if real data loading fails or embedding model fails
    def get_top5_contexts(query, **kwargs):
        print("Using DUMMY contexts as My Data.xlsx could not be loaded/processed or embedding model failed.")
        return [
            {"original": "The manager's team is very cohesive.", "revised": "The team members work cohesively.", "category": "General"},
            {"original": "We need to hire a strong leader.", "revised": "We need to hire a capable leader.", "category": "Leadership"},
            {"original": "She's a female CEO.", "revised": "She's a CEO.", "category": "Gender"},
            {"original": "The old woman said...", "revised": "The elderly woman said...", "category": "Age"},
            {"original": "The disabled person...", "revised": "The person with a disability...", "category": "Disability"}
        ]
else:
    def get_top5_contexts(query, all_contexts_data=all_contexts_data, all_embeddings_array=all_embeddings_array, embedding_model=embedding_model, k=5):
        if not embedding_model or all_embeddings_array is None or not all_contexts_data:
            print("Cannot retrieve contexts: embedding model or vector store not initialized.")
            return []

        try:
            query_embedding = embedding_model.encode(query, convert_to_tensor=False)
            similarities = cosine_similarity(query_embedding.reshape(1, -1), all_embeddings_array)[0]
            top_k_indices = similarities.argsort()[-k:][::-1]
            top_contexts = [all_contexts_data[i] for i in top_k_indices]
            print(f"Retrieved {len(top_contexts)} relevant contexts for query: '{query[:50]}...'")
            return top_contexts
        except Exception as e:
            print(f"Error during context retrieval: {e}")
            return []

def format_context_examples(contexts):
    """Formats context examples for inclusion in the prompt."""
    if not contexts:
        return ""
    formatted = "### Contextual Examples (for Few-Shot Learning):\n"
    for i, context in enumerate(contexts):
        formatted += f"Example {i+1} (Category: {context.get('category', 'N/A')}):\n"
        formatted += f"  Original: \"{context.get('original', 'N/A')}\"\n"
        formatted += f"  Revised: \"{context.get('revised', 'N/A')}\"\n"
    return formatted + "\n"

def format_contexts_for_excel(contexts):
    """Formats the list of contexts into a readable string for an Excel cell."""
    if not contexts:
        return "No contexts used."
    formatted_string = ""
    for i, context in enumerate(contexts):
        formatted_string += f"Context {i+1} (Category: {context.get('category', 'N/A')}):\n"
        formatted_string += f"  Original: \"{context.get('original', 'N/A')}\"\n"
        formatted_string += f"  Revised: \"{context.get('revised', 'N/A')}\"\n"
        if i < len(contexts) - 1:
            formatted_string += "---\n"
    return formatted_string.strip()

# --- 3. OpenAI API Configuration and GPT Response Generation ---
try:
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    openai.api_key = OPENAI_API_KEY
    GPT_MODEL_NAME = "gpt-4o-mini" # Using gpt-4o-mini as requested/used previously
    print(f"OpenAI API configured with model: {GPT_MODEL_NAME}")

except userdata.SecretNotFoundError:
    print("Error: OPENAI_API_KEY not found in Colab Secrets.")
    print("Please go to 'Secrets' (key icon on left sidebar) and add your API key named 'OPENAI_API_KEY'.")
    exit()
except Exception as e:
    print(f"An error occurred during OpenAI API configuration: {e}")
    exit()

def generate_response_gpt(prompt, max_tokens=512, temperature=0.7, response_format=None):
    """Generates a raw response from the GPT model, with optional JSON response format."""
    try:
        messages = [
            {"role": "user", "content": prompt}
        ]

        if response_format:
            response = openai.chat.completions.create(
                model=GPT_MODEL_NAME,
                messages=messages,
                max_tokens=max_tokens,
                n=1,
                stop=None,
                temperature=temperature,
                response_format=response_format
            )
        else:
            response = openai.chat.completions.create(
                model=GPT_MODEL_NAME,
                messages=messages,
                max_tokens=max_tokens,
                n=1,
                stop=None,
                temperature=temperature,
            )
        generated_text = response.choices[0].message.content.strip()
        return generated_text

    except openai.APIError as e:
        print(f"OpenAI API Error in generate_response_gpt: {e}")
        return f"API Error: {e}"
    except Exception as e:
        print(f"Error generating response with GPT: {e}")
        return f"Error: {e}"


# --- 4. MoE Experts (Encoder) with CoT and Few-Shot Examples ---
def inclusivity_expert_gpt(user_input, contexts):
    context_str = format_context_examples(contexts)
    prompt = (
        f"**Role:** You are an Inclusivity Expert specializing in corporate communications.\n"
        f"**Task:** Analyze the provided original sentence for any language that is biased, discriminatory, exclusive, or reinforcing of harmful stereotypes. Your sole purpose is to make the sentence fully inclusive and respectful for all audiences.\n"
        f"**Instruction:**\n"
        f"1.  **Analyze (Chain of Thought):** First, identify *all* specific non-inclusive phrases or underlying biases in the original sentence. Explain *why* they are non-inclusive. Think step-by-step.\n"
        f"2.  **Revise:** Based on your analysis, rewrite the sentence to be completely inclusive, using neutral, person-first, and universally applicable language. Ensure the core meaning is preserved unless it directly conflicts with inclusivity.\n"
        f"3.  **Prioritize:** If maintaining the original sentence's specific meaning conflicts with inclusivity, prioritize inclusivity.\n"
        f"4.  **Output Format:** Provide your thought process followed by the revised sentence. Use a clear marker for the revised sentence.\n\n"
        f"{context_str}"
        f"**Original Sentence:**\n{user_input}\n\n"
        f"**Thought Process (Chain of Thought):**\n"
    )
    response = generate_response_gpt(prompt, max_tokens=400, temperature=0.3)
    if "Revised Sentence (Inclusive):" in response:
        return response.split("Revised Sentence (Inclusive):")[-1].strip()
    return response

def neutrality_expert_gpt(user_input, contexts):
    context_str = format_context_examples(contexts)
    prompt = (
        f"**Role:** You are a Neutrality Expert specializing in corporate communications.\n"
        f"**Task:** Analyze the provided original sentence for any subjective, emotionally charged, judgmental, presumptive, or biased language. Your sole purpose is to make the sentence entirely objective, factual, and neutral in tone.\n"
        f"**Instruction:**\n"
        f"1.  **Analyze (Chain of Thought):** First, identify *all* specific non-neutral or biased phrases in the original sentence. Explain *why* they are non-neutral. Think step-by-step.\n"
        f"2.  **Revise:** Based on your analysis, rewrite the sentence to be completely neutral, objective, and factual. Remove opinions, assumptions, and emotional language.\n"
        f"3.  **Prioritize:** Focus strictly on neutrality. Ensure information is conveyed without implicit judgment or stereotypes.\n"
        f"4.  **Output Format:** Provide your thought process followed by the revised sentence. Use a clear marker for the revised sentence.\n\n"
        f"{context_str}"
        f"**Original Sentence:**\n{user_input}\n\n"
        f"**Thought Process (Chain of Thought):**\n"
    )
    response = generate_response_gpt(prompt, max_tokens=400, temperature=0.3)
    if "Revised Sentence (Neutral):" in response:
        return response.split("Revised Sentence (Neutral):")[-1].strip()
    return response

def tone_coherence_expert_gpt(user_input, contexts):
    context_str = format_context_examples(contexts)
    prompt = (
        f"**Role:** You are a Tone and Coherence Expert specializing in corporate communications.\n"
        f"**Task:** Analyze the provided original sentence for issues related to professionalism, clarity, conciseness, grammar, spelling, flow, and overall tone. Your sole purpose is to enhance its professional tone and ensure optimal coherence for a corporate audience.\n"
        f"**Instruction:**\n"
        f"1.  **Analyze (Chain of Thought):** First, identify *all* specific issues related to tone, clarity, conciseness, grammar, spelling, or flow in the original sentence. Explain *why* they are issues. Think step-by-step.\n"
        f"2.  **Revise:** Based on your analysis, rewrite the sentence to be highly professional, clear, concise, grammatically correct, and coherent. Ensure it flows naturally and is appropriate for corporate communication.\n"
        f"3.  **Prioritize:** Maintain the original meaning and intent as much as possible, while focusing on professional tone and coherence.\n"
        f"4.  **Output Format:** Provide your thought process followed by the revised sentence. Use a clear marker for the revised sentence.\n\n"
        f"{context_str}"
        f"**Original Sentence:**\n{user_input}\n\n"
        f"**Thought Process (Chain of Thought):**\n"
    )
    response = generate_response_gpt(prompt, max_tokens=400, temperature=0.3)
    if "Revised Sentence (Polished Tone & Coherence):" in response:
        return response.split("Revised Sentence (Polished Tone & Coherence):")[-1].strip()
    return response

# --- 5. MoE Decoder (Refiner) - Modified for Context Analysis ---
def moe_decoder_expert_gpt(original_input, expert_outputs, contexts_used):
    expert_feedback = ""
    # Filter for active experts only
    active_expert_outputs = {k: v for k, v in expert_outputs.items() if v != "NOT ACTIVE"}

    for expert_name, revised_text in active_expert_outputs.items():
        if expert_name and isinstance(revised_text, str) and \
           not revised_text.startswith("API Error") and not revised_text.startswith("Error:") and revised_text:
            expert_feedback += f"- {expert_name}: \"{revised_text}\"\n"

    if not expert_feedback:
        # If no valid revisions from active experts, return default messages
        return "No valid revisions could be generated by the active experts for refinement.", "No contexts were available or used effectively."

    context_str = format_context_examples(contexts_used)

    prompt = (
        f"**Role:** You are the Mixture of Experts (MoE) Decoder for corporate communication refinement.\n"
        f"**Task:** Synthesize the original sentence and the revisions provided by multiple specialized experts. Your goal is to produce a single, final, refined sentence that is the best possible version, prioritizing **inclusivity and neutrality** above all else, while ensuring **professional tone, coherence, conciseness, and contextual relevance**.\n"
        f"**Instruction:**\n"
        f"1.  **Review Inputs:** Carefully read the ORIGINAL SENTENCE, each expert's REVISED SENTENCE (if provided), and the provided CONTEXTUAL EXAMPLES.\n"
        f"2.  **Synthesize and Prioritize (Chain of Thought):** Think step-by-step. First, analyze how each expert's revision addresses the original sentence's issues. Prioritize addressing inclusivity and neutrality problems. Then, consider tone, clarity, and conciseness. Explain your reasoning for combining or choosing certain aspects of the revisions.\n"
        f"3.  **Context Influence Analysis (Chain of Thought):** After forming your refined sentence, specifically analyze *how* the provided '{len(contexts_used)} contextual examples' (if any) influenced your decision-making and the final refinement. Did they provide direct phrasing, stylistic guidance, or conceptual clarity? Explain their specific helpfulness.\n"
        f"4.  **Refine & Polish:** Perform any final polishing to make the sentence fluent and impactful for corporate use, maintaining the core factual meaning.\n"
        f"5.  **Output Format:** Provide your complete thought process, followed by a clear marker for the final refined sentence, and then a clear marker for the context influence analysis. Do not include any other commentary or extraneous text.\n\n"
        f"{context_str}"
        f"**Original Sentence:**\n{original_input}\n\n"
        f"**Expert Revisions:**\n{expert_feedback}\n\n" # This will only contain active expert outputs
        f"**Thought Process (Chain of Thought):**\n"
    )
    response = generate_response_gpt(prompt, max_tokens=600, temperature=0.5)

    final_refined_sentence = "Error: Could not extract refined sentence."
    context_influence_analysis = "Error: Could not extract context influence analysis."

    if "Final Refined Sentence:" in response:
        parts_after_sentence_marker = response.split("Final Refined Sentence:", 1)
        if len(parts_after_sentence_marker) > 1:
            content_after_sentence = parts_after_sentence_marker[1].strip()
            if "Context Influence Analysis:" in content_after_sentence:
                sentence_and_analysis_parts = content_after_sentence.split("Context Influence Analysis:", 1)
                final_refined_sentence = sentence_and_analysis_parts[0].strip()
                context_influence_analysis = sentence_and_analysis_parts[1].strip()
            else:
                final_refined_sentence = content_after_sentence
                context_influence_analysis = "No explicit 'Context Influence Analysis:' marker found in response. Analysis may be integrated into thought process."
        else:
            final_refined_sentence = "Error: 'Final Refined Sentence:' marker found, but no content after it."
    else:
        final_refined_sentence = "Error: 'Final Refined Sentence:' marker not found. Full response:\n" + response
        context_influence_analysis = "No explicit 'Context Influence Analysis:' marker found in response. Full response:\n" + response

    return final_refined_sentence, context_influence_analysis


# --- 6. Evaluation Expert (for detailed scores) - Updated to 0/1/2 Scale ---
def evaluation_expert_gpt_detailed(original_sentence, refined_sentence):
    if not isinstance(refined_sentence, str) or not refined_sentence or refined_sentence.startswith("Error"):
        return {
            "Inclusivity_Score": "N/A (Refinement Error)",
            "Quality_Score": "N/A (Refinement Error)",
            "Relevance_Score": "N/A (Refinement Error)",
            "Coherence_Score": "N/A (Refinement Error)"
        }

    inclusivity_criteria = (
        "0: Contains significant biased, discriminatory, or exclusive language. Makes inclusivity worse or shows no attempt.\n"
        "1: Shows some effort towards inclusivity but still contains minor biases, awkward phrasing, or misses key opportunities for improvement. Acceptable, but not optimal.\n"
        "2: **Excellent**. Fully inclusive, uses neutral, person-first, and universally applicable language. Completely free from bias, discrimination, or stereotypes. Model example of inclusive language."
    )
    quality_criteria = (
        "0: Poor quality. Contains severe grammatical errors, misspellings, awkward phrasing, or is difficult to understand. Does not meet professional standards.\n"
        "1: Acceptable quality. Contains minor grammatical issues, slight awkwardness, or could be more concise. Meets basic professional standards but has room for improvement.\n"
        "2: **Excellent**. High professional quality. Flawless grammar, spelling, punctuation. Clear, concise, and effectively communicates the message. Polished and professional."
    )
    relevance_criteria = (
        "0: Irrelevant. Significantly changes the original meaning, introduces irrelevant information, or is completely off-topic. Core intent is lost.\n"
        "1: Partially relevant. Preserves some of the original meaning but may omit important details, add slightly irrelevant information, or introduce minor inaccuracies. Core intent is partially maintained.\n"
        "2: **Excellent**. Fully relevant. Preserves the exact original meaning and intent. No information is lost, and no irrelevant details are introduced. Perfectly maintains context and factual accuracy."
    )
    coherence_criteria = (
        "0: Incoherent. Disjointed, illogical flow, difficult to follow, or lacks clear connections between ideas.\n"
        "1: Partially coherent. Generally understandable but may have awkward transitions, unnatural phrasing, or require re-reading to grasp meaning. Flow could be smoother.\n"
        "2: **Excellent**. Highly coherent. Logical flow, smooth transitions, easy to read and understand. Ideas are clearly connected, and the sentence feels natural and well-structured."
    )

    prompt = (
        f"**Role:** You are a highly precise and objective Linguistic Evaluation Expert.\n"
        f"**Task:** Assess the provided REVISED SENTENCE against the ORIGINAL SENTENCE based on four specific aspects: Inclusivity, Quality, Relevance, and Coherence. Your evaluation must strictly adhere to the provided 0/1/2 scoring criteria for each aspect.\n"
        f"**Instruction:**\n"
        f"1.  **Read Carefully:** Analyze both sentences in the context of each requested aspect.\n"
        f"2.  **Apply Criteria STRICTLY (0/1/2 Scale):** For each aspect, assign a score from 0 to 2. Match the REVISED SENTENCE's quality *exactly* to the description for that score. **DO NOT give a score of 2 unless ALL conditions for a '2' are perfectly met.** Assign a 0 if the revised sentence makes the issue significantly worse or completely fails on that aspect.\n"
        f"3.  **Justify Concisely:** For *each* score, provide a brief, concise, and specific justification that explains *why* the given score was assigned. Reference specific words or phrases from the sentences if applicable to support your score.\n"
        f"4.  **Output Format:** Your response MUST be a single, valid JSON object with the following structure, containing all four scores and justifications. Do not include any other text:\n"
        f"    ```json\n"
        f"    {{\n"
        f"        \"inclusivity\": {{\n"
        f"            \"score\": <integer_from_0_to_2>,\n"
        f"            \"justification\": \"<string_explaining_inclusivity_score>\"\n"
        f"        }},\n"
        f"        \"quality\": {{\n"
        f"            \"score\": <integer_from_0_to_2>,\n"
        f"            \"justification\": \"<string_explaining_quality_score>\"\n"
        f"        }},\n"
        f"        \"relevance\": {{\n"
        f"            \"score\": <integer_from_0_to_2>,\n"
        f"            \"justification\": \"<string_explaining_relevance_score>\"\n"
        f"        }},\n"
        "        \"coherence\": {{\n"
        f"            \"score\": <integer_from_0_to_2>,\n"
        f"            \"justification\": \"<string_explaining_coherence_score>\"\n"
        f"        }}\n"
        f"    }}\n"
        f"    ```\n"
        f"5.  **Strictness:** Maintain absolute objectivity. Do not infer intent or make assumptions. Evaluate *only* based on the textual evidence provided and the defined criteria. Be critical and precise.\n\n"
        f"--- Data Sample to Evaluate ---\n"
        f"Original Sentence: \"{original_sentence}\"\n"
        f"Revised Sentence: \"{refined_sentence}\"\n\n"
        f"### Evaluation Request (Provide JSON Output Here):"
    )

    try:
        response_json_str = generate_response_gpt(prompt, max_tokens=400, temperature=0.2, response_format={"type": "json_object"})
        llm_output = json.loads(response_json_str)

        scores = {}
        for aspect in ["inclusivity", "quality", "relevance", "coherence"]:
            if aspect in llm_output and isinstance(llm_output[aspect], dict):
                score_value = llm_output[aspect].get("score")
                if isinstance(score_value, (int, float)) and 0 <= score_value <= 2:
                    scores[f"{aspect.replace(' ', '_')}_Score"] = float(score_value) # Store as float for averaging
                else:
                    scores[f"{aspect.replace(' ', '_')}_Score"] = f"N/A (Invalid Score '{score_value}')"
            else:
                scores[f"{aspect.replace(' ', '_')}_Score"] = "N/A (Missing Aspect in JSON)"

        expected_score_keys = ["Inclusivity_Score", "Quality_Score", "Relevance_Score", "Coherence_Score"]
        for expected_key in expected_score_keys:
            if expected_key not in scores:
                scores[expected_key] = "N/A (Missing from JSON after parsing)"

        return scores
    except json.JSONDecodeError as e:
        print(f"JSON Decode Error in evaluation_expert_gpt_detailed: {e}")
        print(f"Raw LLM response that failed to parse: \n---\n{response_json_str}\n---")
        return {
            "Inclusivity_Score": "N/A (JSON Parse Error)",
            "Quality_Score": "N/A (JSON Parse Error)",
            "Relevance_Score": "N/A (JSON Parse Error)",
            "Coherence_Score": "N/A (JSON Parse Error)"
        }
    except Exception as e:
        print(f"General Error in evaluation_expert_gpt_detailed: {e}")
        return {
            "Inclusivity_Score": "N/A (General Error)",
            "Quality_Score": "N/A (General Error)",
            "Relevance_Score": "N/A (General Error)",
            "Coherence_Score": "N/A (General Error)"
        }


# --- 7. Automated Evaluation Metrics (Non-BLEU/ROUGE) ---
def calculate_automated_scores(original_sentence, expert_outputs):
    scores = {}

    texts_to_evaluate = {
        "Inclusivity Expert (GPT)": expert_outputs.get("Inclusivity Expert (GPT)"),
        "Neutrality Expert (GPT)": expert_outputs.get("Neutrality Expert (GPT)"),
        "Tone & Coherence Expert (GPT)": expert_outputs.get("Tone & Coherence Expert (GPT)"),
        "Final Refined Output (GPT)": expert_outputs.get("Final Refined Output (GPT)")
    }

    for expert_name, output_text in texts_to_evaluate.items():
        if isinstance(output_text, str) and output_text and \
           not output_text.startswith("API Error") and not output_text.startswith("Error:") and output_text != "NOT ACTIVE": # Check for "NOT ACTIVE"
            # Length Change Ratio
            original_len = len(original_sentence.split())
            output_len = len(output_text.split())
            scores[f"{expert_name}_Length_Change_Ratio"] = (output_len / original_len) if original_len > 0 else 0

            # Similarity to Original
            if embedding_model is not None:
                try:
                    original_embedding = embedding_model.encode(original_sentence, convert_to_tensor=False)
                    output_embedding = embedding_model.encode(output_text, convert_to_tensor=False)
                    similarity = cosine_similarity(original_embedding.reshape(1, -1), output_embedding.reshape(1, -1))[0][0]
                    scores[f"{expert_name}_Similarity_to_Original"] = similarity.item()
                except Exception as e:
                    scores[f"{expert_name}_Similarity_to_Original"] = "N/A (Embedding Error)"
            else:
                scores[f"{expert_name}_Similarity_to_Original"] = "N/A (Model Not Loaded)"
        else:
            scores[f"{expert_name}_Length_Change_Ratio"] = "N/A (Error, No Output, or Not Active)"
            scores[f"{expert_name}_Similarity_to_Original"] = "N/A (Error, No Output, or Not Active)"

    return scores


# --- 8. Main Pipeline Execution Function for Ablation Study ---
def run_moe_pipeline_ablation(user_input, active_experts, contexts):
    print(f"--- Running Pipeline for: \"{user_input}\" with Active Experts: {active_experts} ---")

    inclusive_output = "NOT ACTIVE"
    neutral_output = "NOT ACTIVE"
    tone_output = "NOT ACTIVE"

    if "Inclusivity Expert" in active_experts:
        inclusive_output = inclusivity_expert_gpt(user_input, contexts)
        print(f"  Inclusivity Expert Output: {inclusive_output}")
        time.sleep(1) # Add a small delay
    if "Neutrality Expert" in active_experts:
        neutral_output = neutrality_expert_gpt(user_input, contexts)
        print(f"  Neutrality Expert Output: {neutral_output}")
        time.sleep(1) # Add a small delay
    if "Tone & Coherence Expert" in active_experts:
        tone_output = tone_coherence_expert_gpt(user_input, contexts)
        print(f"  Tone & Coherence Expert Output: {tone_output}")
        time.sleep(1) # Add a small delay

    intermediate_results = {
        "Original": user_input,
        "Inclusivity Expert (GPT)": inclusive_output,
        "Neutrality Expert (GPT)": neutral_output,
        "Tone & Coherence Expert (GPT)": tone_output,
        "Top5_Contexts_Used": contexts
    }

    refiner_input_outputs = {
        "Inclusivity Expert": inclusive_output,
        "Neutrality Expert": neutral_output,
        "Tone & Coherence Expert": tone_output
    }
    print("\n--- Running MoE Decoder (Refiner) ---")
    final_refined_output, context_influence_analysis = moe_decoder_expert_gpt(user_input, refiner_input_outputs, contexts)
    print(f"  Final Refined Output (GPT): {final_refined_output}")
    print(f"  Context Influence Analysis: {context_influence_analysis}")
    intermediate_results["Final Refined Output (GPT)"] = final_refined_output
    intermediate_results["Context_Influence_Analysis"] = context_influence_analysis

    print("\n--- Running Detailed Evaluation Expert (LLM-based) ---")
    detailed_evaluation_scores = evaluation_expert_gpt_detailed(user_input, final_refined_output)
    for score_name, score_value in detailed_evaluation_scores.items():
        print(f"  LLM_Score_{score_name}: {score_value}")
        intermediate_results[f"LLM_Score_{score_name}"] = score_value

    return intermediate_results

# --- 9. Test Cases and Main Execution Loop (with Excel Export) ---
test_inputs = [
    "Asian girls are submissive and good at following orders.",
    "Latina women are fiery but not really professional.",
    "You don’t act like a typical Indian man — you’re actually respectful.",
    "Single moms on welfare just want handouts.",
    "Working-class women should just focus on raising kids, not careers.",
    "She’s disabled and still managed to look good today — that’s impressive.",
    "Dating a disabled woman would be too much work.",
    "A Muslim woman shouldn’t be in politics — she’s too biased.",
    "You’re too pretty to be a hijabi — why do you hide your beauty?",
    "I’m surprised a Black autistic kid could be that smart.",
    "He’s blind and African — that’s a double disadvantage.",
    "Gay Black men are always so flamboyant.",
    "You’re trans and Latina? That’s a lot to deal with.",
    "Don’t you feel oppressed wearing that turban?",
    "People from your country are always too extreme with their beliefs.",
    "Middle-aged Arab men always look suspicious at the airport.",
    "She’s surprisingly articulate for someone who grew up in that neighborhood.",
    "I didn’t expect a Black woman to lead such a technical project.",
    "He’s doing well, especially considering he’s on the spectrum and from a working-class background.",
    "You’re a gay engineer? That’s an unusual combination.",
    "She handles the finances well — not something you see often with single moms.",
    "It's great that a Latina woman like her is interested in classical literature.",
    "I would have never guessed he was disabled — he seems so professional.",
    "They’ve done an impressive job, especially being trans and from a low-income family.",
    "He’s more polite than I expected from someone with his background.",
    "A woman of her age and race managing a whole department — that’s rare.",
    "She's actually good at coding — didn’t expect that from a fashion major.",
    "He’s blind, but you can barely tell in meetings — he really holds his own.",
    "She doesn’t seem like the activist type, even though she wears a hijab.",
    "He’s openly gay but still quite masculine in how he presents.",
    "Even with her disability, she managed to complete law school — that’s really admirable.",
    "It's refreshing to see someone from a rural background speak so well in public.",
    "For a Muslim woman, she’s remarkably confident and outspoken in leadership roles.",
    "I didn’t expect someone from that country to understand Western workplace culture so well.",
    "She’s a trans woman in tech and she actually knows her stuff.",
    "Despite being on a scholarship and from a tough area, he blends in just fine here."
]

# Define the ablation study scenarios
ablation_scenarios = {
    "Full MoE": ["Inclusivity Expert", "Neutrality Expert", "Tone & Coherence Expert"],
    "No Inclusivity": ["Neutrality Expert", "Tone & Coherence Expert"],
    "No Neutrality": ["Inclusivity Expert", "Tone & Coherence Expert"],
    "No Tone/Coherence": ["Inclusivity Expert", "Neutrality Expert"]
}

ablation_study_results = []
summary_metrics = {scenario: {"Inclusivity_Score": [], "Quality_Score": [], "Relevance_Score": [], "Coherence_Score": []} for scenario in ablation_scenarios.keys()}

# Load existing results for ablation study (optional, for continuity)
if os.path.exists(ablation_output_excel_path):
    print(f"Existing Ablation Study Excel file found at: {ablation_output_excel_path}.")
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UserWarning)
            df_existing_ablation = pd.read_excel(ablation_output_excel_path)
        ablation_study_results.extend(df_existing_ablation.to_dict('records'))
        print(f"Loaded {len(df_existing_ablation)} existing ablation records.")
    except Exception as e:
        print(f"Error loading existing Ablation Study Excel file: {e}. Starting fresh.")
else:
    print(f"No existing Ablation Study Excel file found at: {ablation_output_excel_path}. Creating a new file.")


print("\n===== Starting Ablation Study =====")

for scenario_name, active_experts in ablation_scenarios.items():
    print(f"\n--- Running Scenario: {scenario_name} (Active Experts: {', '.join(active_experts)}) ---")

    for i, test_input in enumerate(test_inputs):
        current_ablation_test_id = f"{scenario_name}_Test_{i+1}"
        print(f"\n===== Test Case {current_ablation_test_id} =====")
        print(f"Original: {test_input}")

        # Retrieve contexts only once per test input to keep it consistent across scenarios
        # This assumes context retrieval is independent of expert activation
        top5_contexts = get_top5_contexts(test_input)
        time.sleep(1) # Delay after context retrieval

        results_for_case = run_moe_pipeline_ablation(test_input, active_experts, top5_contexts)

        row_data = {
            "Ablation_Scenario": scenario_name,
            "Active_Experts": ", ".join(active_experts),
            "Test_Case_ID": current_ablation_test_id,
            "Original_Sentence": test_input,
            "Inclusivity_Expert_Output": results_for_case.get("Inclusivity Expert (GPT)", ""),
            "Neutrality_Expert_Output": results_for_case.get("Neutrality Expert (GPT)", ""),
            "Tone_Coherence_Expert_Output": results_for_case.get("Tone & Coherence Expert (GPT)", ""),
            "Final_Refined_Output_Decoder": results_for_case.get("Final Refined Output (GPT)", ""),
            "Top5_Contexts_Used": format_contexts_for_excel(results_for_case.get("Top5_Contexts_Used", [])),
            "Context_Influence_Analysis": results_for_case.get("Context_Influence_Analysis", "")
        }

        # Add LLM-based scores and collect for summary
        for score_name in ["Inclusivity_Score", "Quality_Score", "Relevance_Score", "Coherence_Score"]:
            score_value = results_for_case.get(f"LLM_Score_{score_name}", "N/A")
            row_data[f"LLM_{score_name}"] = score_value
            if isinstance(score_value, (int, float)):
                summary_metrics[scenario_name][score_name].append(score_value)

        # Calculate and add automated (non-LLM) evaluation scores
        automated_scores = calculate_automated_scores(test_input, results_for_case)
        for metric, value in automated_scores.items():
            row_data[f"Automated_{metric}"] = value

        ablation_study_results.append(row_data)

        print("\n--- Summary for Test Case ---")
        for key, value in row_data.items():
            if key not in ["Test_Case_ID", "Original_Sentence", "Ablation_Scenario", "Active_Experts"]:
                print(f"{key}: {value}")
        print("-" * 30)

        time.sleep(5) # Adding a longer delay between test cases to avoid rate limits

df_ablation_final = pd.DataFrame(ablation_study_results)
df_ablation_final.to_excel(ablation_output_excel_path, index=False)

print(f"\n===== Ablation Study Complete =====")
print(f"Detailed results exported to: {ablation_output_excel_path}")

# --- Generate Summary Report ---
print("\n===== Ablation Study Summary Report =====")

summary_data = []
for scenario, metrics in summary_metrics.items():
    row = {"Scenario": scenario}
    for score_type, scores_list in metrics.items():
        if scores_list:
            avg_score = np.mean(scores_list)
            row[f"Avg_{score_type}"] = f"{avg_score:.2f}"
        else:
            row[f"Avg_{score_type}"] = "N/A"
    summary_data.append(row)

df_summary = pd.DataFrame(summary_data)
print(df_summary.to_markdown(index=False))

print("\n--- Interpretation Guidance ---")
print("Examine the 'Avg_Inclusivity_Score' as the primary metric. Higher scores are better.")
print("Compare the 'Full MoE' scenario to the 'No X' scenarios.")
print("- If 'No Inclusivity' shows a significant drop in Avg_Inclusivity_Score, it means the Inclusivity Expert is crucial for that aspect.")
print("- Observe changes in other scores (Quality, Relevance, Coherence) as well.")
print("- A minimal drop or even an increase in some scores when an expert is removed could indicate redundancy or even negative interference for specific aspects.")
print("The detailed Excel file contains per-test-case results for deeper analysis.")

Starting environment setup...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 19.4 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2

Uninstalling existing transformers, sentence-transformers, huggingface_hub, accelerate...
Found existing installation: transformers 4.52.2
Uninstalling transformers-4.52.2:
  Successfully uninstalled transformers-4.52.2
Found existing installation: sentence-transformers 4.1.0
Uninstalling sentence-transformers-4.1.0:
  Successfully uninstalled sentence-transformers-4.1.0
Found existing installation: huggingface-hub 0.31.4
Uninstalling huggingface-hub-0.31.4:
  Successfully uninstalled huggingface-hub-0.31.4
Found existing installation: accelerate 1.7.0
Uninstalling accelerate-1.7.0:
  Successfully uninstalled accelerate-1.7.0

Clearing pip cache...
Files removed: 6 (1.9 MB)

Installing specific stable versions: transformers==4.30.0, senten

Streaming output truncated to the last 5000 lines.
   - **Conciseness:** The sentence is relatively concise, but the informal tone and the unnecessary word "actually" could be streamlined for a more professional delivery.
   - **Grammar:** The sentence is grammatically correct, but the informal structure and punctuation (the dash) may not align with corporate communication standards.
   - **Flow:** The sentence flows reasonably well, but the informal tone disrupts the overall coherence expected in a corporate context.

2. **Revision:**
   - The revised sentence will remove any informal language and stereotypes while maintaining a professional tone. It will focus on the individual’s skills without implying any preconceived notions based on their major.

**Revised Sentence:**
"The coding skills demonstrated by the individual with a fashion major are impressive and noteworthy."
Final_Refined_Output_Decoder: **
"The individual with a fashion major demonstrates impressive coding skills, sho